<a href="https://colab.research.google.com/github/sechali/buku_ajar_ai/blob/main/notebooks/part-07/AI_Modul_7.4_Praktikum_Decision_Tree.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Modul 7.4: Praktikum Pohon Keputusan (Decision Tree CART)

**Mata Kuliah:** Kecerdasan Buatan & Pembelajaran Mesin Terapan  
**Institusi:** Institut Pertanian Stiper (INSTIPER) Yogyakarta  
**Studi Kasus:** Otomatisasi Sortasi Fraksi Kematangan Tandan Buah Segar (TBS) Kelapa Sawit Berbasis Algoritma CART dan Pemangkasan Pohon (*Pruning*)

---

### Capaian Pembelajaran Praktikum
1. Mampu menghitung nilai Entropi Informasi Shannon, Ketidakmurnian Gini, dan Information Gain secara manual melalui komputasi Python.
2. Mampu membangun model Decision Tree menggunakan pustaka Scikit-Learn serta mengekstrak aturan logika eksplisit (*If-Then Rules*).
3. Mampu melakukan pencegahan *overfitting* menggunakan teknik *Pre-Pruning* (`max_depth`) dan *Post-Pruning* (*Cost-Complexity Pruning* via `ccp_alpha`).
4. Mampu mengevaluasi dan memvisualisasikan arsitektur pohon klasifikasi multi-kelas untuk penetapan fraksi mutu TBS di Pabrik Kelapa Sawit.

## 1. Import Pustaka & Konfigurasi Lingkungan

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Pengaturan visualisasi
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.size'] = 10
plt.rcParams['figure.dpi'] = 120

print("[OK] Seluruh pustaka pohon keputusan berhasil dimuat!")

## 2. Pembangkitan Dataset Sortasi TBS Pabrik Kelapa Sawit

Parameter inspeksi sortasi:
- `Brondolan_Persen`: Persentase brondolan lepas dari tandan (%)
- `FFA_Persen`: Kadar Asam Lemak Bebas brondolan (%)
- `Berat_Tandan_kg`: Berat rata-rata tandan (kg)
- `Fraksi_Mutu`: Label kelas (0: Mentah Afkir, 1: Kurang Matang, 2: Matang Standar, 3: Matang Prima)

In [ ]:
np.random.seed(42)
n_samples = 280

brondolan = np.random.uniform(1.0, 55.0, n_samples)
ffa = np.random.uniform(1.0, 6.5, n_samples)
berat = np.random.uniform(8.0, 38.0, n_samples)

# Logika agronomi sortasi PKS realistis
labels = []
for b, f in zip(brondolan, ffa):
    if b <= 8.5:
        labels.append(0)  # Mentah Afkir (brondolan sangat sedikit)
    elif b <= 18.0:
        labels.append(1 if f <= 3.4 else 0)  # Kurang Matang atau Rusak Asam
    elif b <= 40.0:
        labels.append(3 if f <= 2.6 else 2)  # Matang Prima vs Matang Standar
    else:
        labels.append(2 if f <= 3.8 else 0)  # Lewat Matang / Busuk

df_tbs = pd.DataFrame({
    'Brondolan_Persen': np.round(brondolan, 1),
    'FFA_Persen': np.round(ffa, 2),
    'Berat_Tandan_kg': np.round(berat, 1),
    'Fraksi_Mutu': labels
})

print("Distribusi Sampel Fraksi Kematangan TBS:")
print(df_tbs['Fraksi_Mutu'].value_counts())
display(df_tbs.head())

## 3. Eksplorasi Visual: Sebaran Data antar Fraksi Mutu

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
palette = {0: '#d32f2f', 1: '#f57c00', 2: '#fbc02d', 3: '#388e3c'}
names_map = {0: 'Mentah (0)', 1: 'Kurang Matang (1)', 2: 'Standar (2)', 3: 'Prima (3)'}
df_plot = df_tbs.copy()
df_plot['Nama_Fraksi'] = df_plot['Fraksi_Mutu'].map(names_map)

sns.scatterplot(
    data=df_plot,
    x='Brondolan_Persen',
    y='FFA_Persen',
    hue='Nama_Fraksi',
    palette=['#d32f2f', '#f57c00', '#fbc02d', '#388e3c'],
    s=70,
    edgecolor='k',
    alpha=0.85,
    ax=ax
)
ax.set_title("Sebaran Mutu TBS Berdasarkan Persentase Brondolan & Kadar FFA", fontweight='bold')
ax.set_xlabel("Persentase Brondolan Lepas (%)")
ax.set_ylabel("Kadar Asam Lemak Bebas / FFA (%)")
plt.show()

## 4. Implementasi Manual Metrik Ketidakmurnian (Gini & Entropi)

Formula matematis:
$$H(D) = -\sum_{c=1}^C p_c \log_2(p_c)$$
$$I_G(D) = 1 - \sum_{c=1}^C p_c^2$$

In [ ]:
def hitung_entropi(y: np.ndarray) -> float:
    _, counts = np.unique(y, return_counts=True)
    probabilities = counts / len(y)
    # Hilangkan p=0 untuk menghindari log2(0)
    probabilities = probabilities[probabilities > 0]
    return -np.sum(probabilities * np.log2(probabilities))

def hitung_gini(y: np.ndarray) -> float:
    _, counts = np.unique(y, return_counts=True)
    probabilities = counts / len(y)
    return 1.0 - np.sum(probabilities ** 2)

y_total = df_tbs['Fraksi_Mutu'].values
print("=== METRIK KETIDAKMURNIAN SIMPUL INDUK POPULASI TBS ===")
print(f"Entropi Shannon Induk : {hitung_entropi(y_total):.4f} bit")
print(f"Indeks Gini Induk     : {hitung_gini(y_total):.4f}")

## 5. Partisi Data & Pelatihan Pohon Keputusan Penuh (Unpruned)

In [ ]:
fitur = ['Brondolan_Persen', 'FFA_Persen', 'Berat_Tandan_kg']
target = 'Fraksi_Mutu'

X = df_tbs[fitur]
y = df_tbs[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# Model 1: Pohon Penuh (Tanpa batasan kedalaman - Berpotensi Overfitting)
tree_full = DecisionTreeClassifier(criterion='gini', random_state=42)
tree_full.fit(X_train, y_train)

acc_train_full = accuracy_score(y_train, tree_full.predict(X_train))
acc_test_full = accuracy_score(y_test, tree_full.predict(X_test))

print("=== EVALUASI POHON KEPUTUSAN PENUH (UNPRUNED) ===")
print(f"Kedalaman Pohon (Depth)     : {tree_full.get_depth()} tingkat")
print(f"Jumlah Simpul Daun (Leaves) : {tree_full.get_n_leaves()} daun")
print(f"Akurasi Data Latih          : {acc_train_full*100:.2f}%")
print(f"Akurasi Data Uji            : {acc_test_full*100:.2f}%")
print(f"Gap Overfitting             : {(acc_train_full - acc_test_full)*100:.2f}%")

## 6. Pre-Pruning: Eksplorasi Kedalaman Maksimum (`max_depth`)

Kita membatasi kedalaman pohon dari 1 hingga 10 untuk melihat titik kompromi bias-varians optimal.

In [ ]:
depths = range(1, 11)
train_scores = []
test_scores = []

for d in depths:
    clf = DecisionTreeClassifier(max_depth=d, random_state=42)
    clf.fit(X_train, y_train)
    train_scores.append(accuracy_score(y_train, clf.predict(X_train)))
    test_scores.append(accuracy_score(y_test, clf.predict(X_test)))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(depths, np.array(train_scores)*100, color='#1f77b4', marker='o', linewidth=2, label='Akurasi Data Latih')
ax.plot(depths, np.array(test_scores)*100, color='#d62728', marker='s', linewidth=2.5, label='Akurasi Data Uji (Validasi)')

best_depth = depths[np.argmax(test_scores)]
ax.axvline(best_depth, color='#2ca02c', linestyle='--', linewidth=2, label=f'Kedalaman Optimal (Depth = {best_depth})')
ax.set_title("Eksplorasi Pre-Pruning: Pengaruh Max Depth terhadap Akurasi", fontweight='bold')
ax.set_xlabel("Parameter max_depth")
ax.set_ylabel("Akurasi (%)")
ax.legend()
plt.show()

print(f"Kedalaman Pre-Pruning Terbaik: max_depth = {best_depth}")

## 7. Post-Pruning: Minimal Cost-Complexity Pruning (`ccp_alpha`)

Fungsi penalti kompleksitas:
$$R_\alpha(T) = R(T) + \alpha |T|$$

In [ ]:
path = tree_full.cost_complexity_pruning_path(X_train, y_train)
ccp_alphas, impurities = path.ccp_alphas, path.impurities

clfs_pruning = []
for alpha in ccp_alphas[:-1]:  # Abaikan alpha maksimum yang memangkas ke simpul akar tunggal
    clf = DecisionTreeClassifier(random_state=42, ccp_alpha=alpha)
    clf.fit(X_train, y_train)
    clfs_pruning.append(clf)

scores_pruning = [accuracy_score(y_test, clf.predict(X_test)) for clf in clfs_pruning]
best_idx = np.argmax(scores_pruning)
best_alpha = ccp_alphas[best_idx]
best_pruned_tree = clfs_pruning[best_idx]

print("=== OPTIMASI MINIMAL COST-COMPLEXITY PRUNING ===")
print(f"Nilai ccp_alpha Terbaik   : {best_alpha:.5f}")
print(f"Kedalaman Pohon Terpangkas: {best_pruned_tree.get_depth()} tingkat")
print(f"Jumlah Daun Terpangkas    : {best_pruned_tree.get_n_leaves()} daun")
print(f"Akurasi Data Uji          : {accuracy_score(y_test, best_pruned_tree.predict(X_test))*100:.2f}%")

## 8. Visualisasi Arsitektur Pohon Keputusan Terpangkas

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7), dpi=150)
plot_tree(
    best_pruned_tree,
    feature_names=fitur,
    class_names=['Mentah', 'Krg Matang', 'Standar', 'Prima'],
    filled=True,
    rounded=True,
    fontsize=8,
    ax=ax
)
ax.set_title("Struktur Pohon Keputusan Terpangkas Optimal (CART)", fontweight='bold', pad=12)
plt.show()

## 9. Ekstraksi Aturan Logika Eksplisit (*If-Then Business Rules*)

In [ ]:
rules = export_text(best_pruned_tree, feature_names=fitur)
print("=== DAFTAR ATURAN LOGIKA BISNIS SORTASI TBS PKS ===\n")
print(rules)

## 10. Evaluasi Akhir: Matriks Konfusi & Laporan Klasifikasi

In [ ]:
y_pred_opt = best_pruned_tree.predict(X_test)
cm = confusion_matrix(y_test, y_pred_opt)

fig, ax = plt.subplots(figsize=(7, 5.5))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='YlGnBu',
    xticklabels=['Mentah', 'Krg Matang', 'Standar', 'Prima'],
    yticklabels=['Mentah', 'Krg Matang', 'Standar', 'Prima'],
    ax=ax
)
ax.set_title("Matriks Konfusi Pohon Keputusan Terpangkas", fontweight='bold')
ax.set_xlabel("Prediksi Sistem Sortasi")
ax.set_ylabel("Aktual Laboratorium")
plt.show()

print("=== LAPORAN EVALUASI MODEL CART TERPANGKAS ===")
print(classification_report(
    y_test, y_pred_opt,
    target_names=['Mentah', 'Krg Matang', 'Standar', 'Prima']
))

## 11. Tugas Mandiri & Eksplorasi Mahasiswa

1. Uji sebuah truk TBS baru dengan parameter: Brondolan = 25.0%, FFA = 2.1%, Berat Tandan = 22.0 kg. Berdasarkan pohon keputusan di atas, masuk ke fraksi mutu manakah truk tersebut?
2. Ubah kriteria pemisahan dari `criterion='gini'` menjadi `criterion='entropy'`. Apakah menghasilkan struktur pohon yang identik atau berbeda? Mengapa demikian?
3. Jelaskan mengapa model pohon keputusan tunggal sering kali memiliki varians tinggi (*high variance*) dan bagaimana metode ensemble seperti Random Forest mengatasi masalah tersebut.